# Лабораторная работа № 2

## Группа вращений SO(3): экспоненциальное отображение, кватернионы, интерполяция

**Дисциплина:** Введение в технологии виртуальной дополненной реальности и робототехники
**Тема курса:** 3. Группа вращений SO(3): матрицы, углы Эйлера, экспоненциальное отображение, кватернионы

---

**Студент:** _фамилия, имя, группа_
**Дата:**

---

### Цель работы

Освоить три способа представления ориентации — матрицу, вектор поворота и кватернион, —
реализовать переходы между ними, обнаружить численно вырождение углов Эйлера
и сравнить представления по устойчивости при длительном интегрировании.

### Что понадобится

`numpy`, `matplotlib`. Из лабораторной работы № 1 переиспользуются `rotx`, `roty`, `rotz`.

### Правило практикума

Алгоритм вы реализуете сами. `scipy.spatial.transform.Rotation` допустимо использовать
**только** для сверки — и такую сверку нужно показать явно.

## Теоретическая справка

### Экспоненциальное отображение

Всякий поворот есть поворот вокруг некоторой оси (теорема Эйлера). Если $\hat\omega$ —
единичный вектор оси, $\theta$ — угол, то поворот задаётся **вектором поворота**
$\omega = \theta\hat\omega \in \mathbb{R}^3$, а матрица получается по формуле Родрига:

$$R = \exp([\omega]_\times) = I + \sin\theta\,[\hat\omega]_\times
  + (1-\cos\theta)\,[\hat\omega]_\times^2,$$

где $[\cdot]_\times$ — кососимметрическая матрица векторного произведения. Обратное
отображение:

$$\theta = \arccos\frac{\operatorname{tr}R - 1}{2},
\qquad
[\hat\omega]_\times = \frac{R - R^{\mathsf T}}{2\sin\theta}.$$

Множество кососимметрических матриц $3\times3$ — это алгебра Ли $\mathfrak{so}(3)$,
касательное пространство к $SO(3)$ в единице. Она трёхмерна, и это ещё одно объяснение
того, почему у поворота три параметра.

**Особые случаи, которые придётся обработать явно:** $\theta \to 0$ (деление на
$\sin\theta$) и $\theta \to \pi$ (матрица $R - R^{\mathsf T}$ обращается в нуль, ось
приходится извлекать из $R + I$).

### Кватернионы

Единичный кватернион $q = (w, x, y, z)$, $\|q\| = 1$, задаёт поворот на угол $\theta$
вокруг оси $\hat\omega$ как

$$q = \left(\cos\tfrac{\theta}{2},\; \hat\omega \sin\tfrac{\theta}{2}\right).$$

Композиция поворотов — произведение кватернионов; обращение — сопряжение.
Отображение $q \mapsto R(q)$ является **двойным накрытием**: $q$ и $-q$ задают один
и тот же поворот. Отсюда практическое следствие для интерполяции: перед началом нужно
выбрать знак второго кватерниона так, чтобы скалярное произведение было неотрицательным,
иначе интерполяция пойдёт «длинным путём».

### Сферическая линейная интерполяция

$$\mathrm{slerp}(q_0, q_1, t) =
\frac{\sin((1-t)\Omega)\,q_0 + \sin(t\Omega)\,q_1}{\sin\Omega},
\qquad \cos\Omega = q_0 \cdot q_1 .$$

Это движение по дуге большого круга на сфере $S^3$ с постоянной угловой скоростью.
Покомпонентная интерполяция углов Эйлера такого свойства не даёт — в этом вы убедитесь
численно в задании 5.

### Вырождение углов Эйлера

Связь производных углов с угловой скоростью для последовательности ZYX имеет вид
$\omega = E(\psi,\theta,\varphi)\,(\dot\psi, \dot\theta, \dot\varphi)^{\mathsf T}$, причём
$\det E = -\cos\theta$. При $\theta \to \pm 90^\circ$ матрица вырождается: две оси
поворота совпадают, теряется степень свободы, а обратная задача становится
неограниченно плохо обусловленной. Это и есть **gimbal lock**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# промежуток для рандомных чисел векторов сдвига
SHIFT_MAX: int = 5
SHIFT_MIN: int = -SHIFT_MAX
SEED = 0

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(seed=SEED)

# константа как в UE5
SMALL_NUMBER = 1e-8

# ue_is_nearly_equal(val: float) -> bool -- копирка bool IsNearlyEqual(float, float) из UE5
def ue_is_nearly_equal(a: float, b: float) -> bool:
	return abs(a - b) <= SMALL_NUMBER

# ue_is_nearly_zero(val: float) -> bool -- копирка bool IsNearlyZero(float, float) из UE5
def ue_is_nearly_zero(val: float) -> bool:
	return abs(val) <= SMALL_NUMBER

def ue_is_nearly_pi(val: float) -> bool:
	return abs(val - np.pi) <= 1e-4


def rotx(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

def roty(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def rotz(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

# скопировано из первой лабораторки (из random_transform)
def rand_R() -> np.asarray:
	a = rng.uniform(-np.pi, np.pi, size=3)
	return rotx(a[0]) @ roty(a[1]) @ rotz(a[2])

## Задание 1. Экспоненциальное и логарифмическое отображения

Реализуйте `skew(w)`, `unskew(K)`, `so3_exp(w)` (формула Родрига) и `so3_log(R)`.

**Обязательно обработайте особые случаи** $\theta \approx 0$ и $\theta \approx \pi$ —
без этого проверка не пройдёт на случайной выборке.

**Контроль.** Для 300 случайных матриц поворота максимум $\|\exp(\log R) - R\|$
должен быть меньше $10^{-9}$. Отдельно проверьте $\log(\exp \omega) = \omega$.

In [ ]:
# формулу нашёл в интернете (vmath - https://vmath.ru/vf5/algebra2/skewsym)
def skew(w):
	x, y, z = w[0], w[1], w[2]
	return np.asarray(
		[[0, -z,  y],
		[ z,  0, -x],
		[-y,  x,  0]])

def unskew(K):
	return np.asarray([K[2,1], K[0,2], K[1,0]])

# по формуле родрига R = exp(w) = E + sin(a) * sw + (1 - cos(a)) * (sw @ sw) 
# a - угол, то же, что и длина вектора w
# sw - кососимметрическая матрица, полученная через skew(w)
# где E - единичная матрица
def so3_exp(w):
	a = np.linalg.norm(w)
	w = skew(w)

	# т.к. в формуле родрига w нужно разделить на угол:
	# когда a стремится к нулю
	if ue_is_nearly_zero(a):
		# a -> 0: sin(a) = a; cos(a) = 1 - a*a/2 => 1 - cos(a) = a*a/2 или = 2sin(a/2)^2
		# в таком случае получаем a*w/a, а не cos(a)*w/a
		# то же и для cos => (w@w)/2
		return np.eye(3) + w + 0.5 * (w @ w)
	# все остальные случаи
	w /= a
	return np.eye(3) + np.sin(a) * w + (1 - np.cos(a)) * (w @ w)

# a = arccos((tr R - 1) / 2)
# sw = (R - R.T) / 2sin(a)
def so3_log(R: np.asarray):
	a = np.arccos((np.trace(R) - 1) / 2)
	
	if ue_is_nearly_zero(a):
		# т.к. при a -> 0: sin(a) = a
		# это то же, что и a * unskew((trR - 1) / 2 * a)
		# a и a сокращаются и остаётся:
		return unskew((np.trace(R) - 1) / 2)

	if ue_is_nearly_pi(a):
		nnt = (R + np.eye(3)) / 2
		i = np.argmax(np.diag(nnt))
		n = nnt[:i] / np.sqrt(nnt[i,i])
		n /= np.linalg.norm(n)
		if np.dot((R - R.T) / 2, n) < 0: n = -n
		return a * n

	# общий случай (по формуле)
	return a * unskew((R - R.T) / (2 * np.sin(a)))

def task_1():
	err_count = 0
	err_max_delta = 0
	for i in range(300):
		R = rand_R()
		delta = np.linalg.norm(so3_exp(so3_log(R)) - R)
		if delta > err_max_delta: err_max_delta = delta
		if delta > 1e-9:
			err_count += 1
	print(f"||exp(log(R)) - R||\nmax delta = {err_max_delta}\nerr count = {err_count}")

task_1()

## Задание 2. Переходы между представлениями

Реализуйте `R_to_quat`, `quat_to_R`, `R_to_euler_zyx`, `euler_zyx_to_R`.

Для `R_to_quat` используйте устойчивый алгоритм: выбор ветви по наибольшему
диагональному элементу. Наивная формула через $\sqrt{1+\operatorname{tr}R}$
теряет точность при $\theta$ около $\pi$ — проверьте это сами и покажите в отчёте.

**Контроль.** Круговые проверки на 300 случайных матрицах: погрешность порядка $10^{-15}$
для кватернионов и не хуже $10^{-10}$ для углов Эйлера.

**Вопрос для вывода:** почему точность прохода через углы Эйлера систематически хуже?

In [ ]:
def R_to_quat(R):
	# нашёл в интернете формулу и нудно писал всё по ней

	t = np.trace(R)
	i = int(np.argmax([R[0, 0], R[1, 1], R[2, 2], t]))

	# в случае w 
	if i == 3:
		s = 2 * np.sqrt(1 + t)
		w = s / 4
		x = (R[2, 1] - R[1, 2]) / s
		y = (R[0, 2] - R[2, 0]) / s
		z = (R[1, 0] - R[0, 1]) / s
	# в случае x
	elif i == 0:
		s = 2 * np.sqrt(1 + R[0, 0] - R[1, 1] - R[2, 2])
		x = s / 4
		w = (R[2, 1] - R[1, 2]) / s
		y = (R[0, 1] + R[1, 0]) / s
		z = (R[0, 2] + R[2, 0]) / s
	# в случае y
	elif i == 1:
		s = 2 * np.sqrt(1 - R[0, 0] + R[1, 1] - R[2, 2])
		y = s / 4
		w = (R[0, 2] - R[2, 0]) / s
		x = (R[0, 1] + R[1, 0]) / s
		z = (R[1, 2] + R[2, 1]) / s
	# в случае z
	else:
		s = 2 * np.sqrt(1 - R[0, 0] - R[1, 1] + R[2, 2])
		z = s / 4
		w = (R[1, 0] - R[0, 1]) / s
		x = (R[0, 2] + R[2, 0]) / s
		y = (R[1, 2] + R[2, 1]) / s
	
	q = np.array([w, x, y, z])
	q /= np.linalg.norm(q)
	
	return q if q[0] >= 0 else -q

def quat_to_R(q):
	q /= np.linalg.norm(q)
	w, x, y, z = q[0], q[1], q[2], q[3]

	return np.array(
		[[1 - 2 * (y*y + z*z),	2 * (x*y - w*z),		2 * (x*z + w*y)],
		[2 * (x*y + w*z),		1 - 2 * (x*x + z*z),	2 * (y*z - w*x)],
		[2 * (x*z - w*y), 		2 * (y*z + w*x),		1 - 2 * (x*x + y*y)]])

def R_to_euler_zyx(R):
	return np.array(
		[np.atan2(R[1,0], R[0,0]),
		np.arcsin(-R[2,0]),
		np.atan2(R[2,1], R[2,2])])

def euler_zyx_to_R(a):
	return rotz(a[0]) @ roty(a[1]) @ rotx(a[2])


def task_2():
	quat_err_max_delta = 0
	quat_err_count = 0
	euler_err_max_delta = 0
	euler_err_count = 0
	
	for i in range(300):
		R = rand_R()
		quat_delta = np.linalg.norm(quat_to_R(R_to_quat(R)) - R)
		euler_delta = np.linalg.norm(euler_zyx_to_R(R_to_euler_zyx(R)) - R)

		if quat_delta > quat_err_max_delta: quat_err_max_delta = quat_delta
		if quat_delta >= 1e-15: quat_err_count += 1 
		
		if euler_delta > euler_err_max_delta: euler_err_max_delta = euler_delta
		if euler_delta >= 1e-10: euler_err_count += 1 

	print(f"Кватернионы:\n\tmax_err_delta = {quat_err_max_delta}\n\terr_count = {quat_err_count}\nУглы Эйлера:\n\tmax_err_delta = {euler_err_max_delta}\n\terr_count = {euler_err_count}")

tasl_2()

## Задание 3. Двойное накрытие

Покажите численно, что $q$ и $-q$ дают одну и ту же матрицу поворота.

**Вопрос для вывода:** какое практическое затруднение создаёт двойное накрытие
при интерполяции и при усреднении ориентаций? Как его снимают?

In [ ]:
def task_3():
	q = rand_vector(size=4)
	print(f'Кватернион: q = {q}\n')
	print(f'R из q:\n{quat_to_R(q)}\n')
	print(f'R из -q:\n{quat_to_R(-q)}\n')
	print(f'разница между R из q и -q = {np.linalg.norm(quat_to_R(q) - quat_to_R(-q))}')

task_3()

## Задание 4. Gimbal lock — численное обнаружение

Реализуйте `euler_rate_matrix(a)` — матрицу $E$ из теоретической справки.

Постройте таблицу числа обусловленности $E$ и модуля определителя для тангажа
$0^\circ, 30^\circ, 60^\circ, 80^\circ, 89^\circ, 89{,}9^\circ, 89{,}99^\circ$.
Постройте график зависимости числа обусловленности от тангажа.

**Вопросы для вывода:** чему равен $|\det E|$ аналитически и совпадает ли это
с численным результатом? Во сколько раз растёт число обусловленности при уменьшении
угла до вырождения в десять раз? Что физически происходит с механизмом в этот момент?

In [ ]:
def euler_rate_matrix(a):
	return np.array(
		[[0, -np.sin(a[0]), np.cos(a[1])*np.cos(a[0])],
		[0, np.cos(a[0]), np.cos(a[1])*np.sin(a[0])],
		[1, 0, - np.sin(a[1])]])

def task_4():
	print('\t|   тангаж\t|   cond\t|   det\t\t|   cos\t\t|')
	for i in [0, 30, 60, 80, 89.9, 89.99]:
		E = euler_rate_matrix((0, np.radians(i), 0))
		cond = np.linalg.cond(E)
		print('\t|   %.2f\t|   %.3f\t|   %.2e\t|   %.2e\t|' % (i, cond, abs(np.linalg.det(E)), np.radians(i)))

task_4()

## Задание 5. SLERP против интерполяции углов Эйлера

Реализуйте `slerp(q0, q1, t)` с выбором кратчайшего пути и с линейной ветвью
для близких кватернионов.

Возьмите два поворота, постройте интерполяцию 400 шагами двумя способами: SLERP
и покомпонентная линейная интерполяция углов Эйлера. Для каждой траектории вычислите
модуль угловой скорости численно:
$|\omega_i| = \|\log(R_{i+1} R_i^{\mathsf T})\| / \Delta t$. Постройте оба профиля
на одном графике.

**Вопросы для вывода:** какой разброс угловой скорости у каждого метода? Почему у SLERP
он на уровне машинной погрешности? Где в VR-шлеме это свойство критично?

In [ ]:
# TODO: задание 5

## Задание 6. Дрейф при длительном интегрировании

Задайте постоянную угловую скорость и шаг $\Delta t = 10^{-3}$. Выполните $2\cdot10^5$
шагов тремя способами: накопление матрицы, накопление кватерниона без нормировки,
накопление кватерниона с нормировкой на каждом шаге.

Измеряйте $\|R^{\mathsf T}R - I\|$ и $\bigl|\,\|q\| - 1\bigr|$ в 40 промежуточных точках
и постройте график в двойном логарифмическом масштабе.

**Вопросы для вывода:** во сколько раз матрица «портится» сильнее кватерниона и почему?
Почему нормировка кватерниона убирает дрейф полностью, а простого аналога для матрицы нет?
За какое время реального полёта при частоте 1000 Гц набегает $2\cdot10^5$ шагов?

In [ ]:
# TODO: задание 6

## Задание 7. Цена восстановления: SVD против нормировки

Измерьте время SVD-ортогонализации матрицы и время нормировки кватерниона
(по 20000 повторений каждое).

**Вопрос для вывода:** соедините результат с заданием 6 и сформулируйте, в каких
задачах ориентацию следует хранить кватернионом, а в каких матрицей.

In [ ]:
# TODO: задание 7

## Выводы

_Ответьте на все вопросы, отмеченные в заданиях как «вопрос для вывода», приводя
конкретные числа, которые вы получили._

## Контрольные вопросы

1. Сформулируйте теорему Эйлера о вращении. Как из неё следует, что у поворота три параметра?
2. Выведите формулу Родрига или объясните структуру её слагаемых.
3. Почему в `so3_log` нужно отдельно обрабатывать $\theta \approx 0$ и $\theta \approx \pi$?
4. Что такое двойное накрытие и какие практические затруднения оно создаёт?
5. Почему кватернион задаёт поворот через половинный угол?
6. Что такое gimbal lock? Приведите математическое и механическое объяснение.
7. Почему нельзя интерполировать ориентации покомпонентной интерполяцией углов Эйлера?
8. Почему сумма двух матриц поворота не является матрицей поворота? Как тогда усреднять ориентации?
9. Сравните три представления ориентации по памяти, стоимости композиции и устойчивости.
10. Где в VR-шлеме используется SLERP и что произойдёт при отказе от него?